In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.getcwd())
if os.path.basename(PROJECT_ROOT) == "notebooks":
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [2]:
import torch
import torch.nn as nn
from safetensors.torch import load_file
from torchmetrics.text import WordErrorRate
from transformers import TrainingArguments

from srcs.datasets.vicocktail import Collator, load_vicocktail
from srcs.nets.backend.nets_utils import make_non_pad_mask
from srcs.nets.backend.refiner.refiner import MyRefiner
from srcs.nets.e2e import get_model as create_model
from srcs.nets.lora import apply_lora
from srcs.nets.utils import ctc_decode, freeze, load_weights
from srcs.spm.spm_train import ensure_unigram
from srcs.spm.text_transofm import TextTransform
from srcs.trainer.trainer import HFTrainer

d:\projects\VietnameseVSR\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
OUTPUT_PATH = os.path.join(PROJECT_ROOT, "checkpoints", "refiner_p_only")
REFINER_CHECKPOINT = os.path.join(OUTPUT_PATH, "checkpoint-105894")
BASE_MODEL_CHECKPOINT = os.path.join(
    PROJECT_ROOT, "checkpoints", "vsr_lora_6", "final"
)

SEED = 42
NUM_WORKERS = 0
BATCH = 12

In [4]:
dataset_splits = load_vicocktail(
    test_fraction=1.0,
    splits=("test",),
    seed=SEED,
)
test_dataset = dataset_splits["test"]

print(f"Test samples: {len(test_dataset):,}")

Test samples: 1,167


In [5]:
model_path, units_path = ensure_unigram()
text_transform = TextTransform(model_path, units_path)

In [6]:
def load_vsr_lora(vocab_size, checkpoint_path, model_size="small"):
    model = create_model("auto-vsr", vocab_size, size=model_size)
    apply_lora(
        model=model,
        start_block=0,
        rank=16,
        alpha=16,
        dropout_rate=0.05,
        target_modules=("linear_q", "linear_v"),
    )
    report = load_weights(model, checkpoint_path)
    freeze(model)
    print(f"Loaded base model tensors: {report['loaded']:,}")
    return model

In [7]:
class CTCOnlyRefinerModel(nn.Module):
    def __init__(self, model, vocab_size, k=0):
        super().__init__()
        self.model = model
        self.refiner = MyRefiner(vocab_size=vocab_size, k=k)
        freeze(self.model)

    def train(self, mode=True):
        super().train(mode)
        self.model.eval()
        return self

    def forward(self, videos, video_lengths, labels=None, label_lengths=None):
        with torch.no_grad():
            contexts = self.model.get_contexts(videos, video_lengths)
            base_logits = contexts["logits"]

        mask = make_non_pad_mask(contexts["input_lengths"]).to(videos.device)
        outputs = self.refiner(logits=base_logits, visual_feats=None, mask=mask)
        new_logits = outputs["logits_steps"][-1]

        loss = self.model.ctc.loss_from_logits(
            new_logits, contexts["input_lengths"], labels, label_lengths
        )

        return {
            "loss": loss,
            "logits": new_logits,
            "input_lengths": contexts["input_lengths"],
            "base_logits": base_logits,
        }

In [8]:
class CustomTrainer(HFTrainer):
    def __init__(self, *args, text_transform, **kwargs):
        super().__init__(*args, **kwargs)
        self.text_transform = text_transform
        self.output_metrics = {"train": self._metrics(), "eval": self._metrics()}

    @staticmethod
    def _metrics():
        return {
            "wer": WordErrorRate(),
            "base_wer": WordErrorRate(),
            "sample_count": 0,
            "flip_count": 0,
            "frame_count": 0,
        }

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        loss, outputs = super().compute_loss(
            model=model,
            inputs=inputs,
            return_outputs=True,
            num_items_in_batch=num_items_in_batch,
        )

        mode = "train" if model.training else "eval"
        with torch.no_grad():
            self._update_metrics(outputs, inputs, mode)

        if return_outputs:
            return loss, outputs

        return loss

    def _update_metrics(self, outputs, inputs, mode):
        new_logits = outputs["logits"]
        base_logits = outputs["base_logits"]
        input_lengths = outputs["input_lengths"]
        valid_mask = make_non_pad_mask(input_lengths).to(new_logits.device)

        state = self.output_metrics[mode]
        frame_flips = base_logits.argmax(dim=-1) != new_logits.argmax(dim=-1)

        state["flip_count"] += int(frame_flips[valid_mask].sum().item())
        state["frame_count"] += int(valid_mask.sum().item())

        references = self._decode_references(inputs)
        hypotheses = self._decode_logits(new_logits, input_lengths)
        state["wer"].update(hypotheses, references)
        state["sample_count"] += len(references)

        if mode == "eval":
            base_hypotheses = self._decode_logits(base_logits, input_lengths)
            state["base_wer"].update(base_hypotheses, references)

    def _decode_logits(self, logits, input_lengths):
        token_ids = ctc_decode(logits, input_lengths, self.text_transform.blank_id)
        return [self.text_transform.decode(ids) for ids in token_ids]

    def _decode_references(self, inputs):
        labels = inputs["labels"].detach().cpu()
        label_lengths = inputs["label_lengths"].detach().cpu().tolist()
        return [
            self.text_transform.decode(label[:length])
            for label, length in zip(labels, label_lengths)
        ]

    def log(self, logs, start_time=None):
        if "eval_loss" in logs:
            self._add_output_metrics(logs, "eval")
        elif "loss" in logs or "train_loss" in logs:
            self._add_output_metrics(logs, "train")

        super().log(logs, start_time=start_time)

    def _add_output_metrics(self, logs, mode):
        state = self.output_metrics[mode]
        if state["sample_count"] == 0:
            return

        prefix = "eval_" if mode == "eval" else ""
        logs[prefix + "wer"] = state["wer"].compute().item()
        logs[prefix + "refined_flip_rate"] = (
            state["flip_count"] / state["frame_count"]
        )

        if mode == "eval":
            logs["eval_base_wer"] = state["base_wer"].compute().item()

        self.output_metrics[mode] = self._metrics()

In [9]:
base_model = load_vsr_lora(text_transform.vocab_size, BASE_MODEL_CHECKPOINT)
model = CTCOnlyRefinerModel(
    model=base_model,
    vocab_size=text_transform.vocab_size,
    k=0,
)

checkpoint_path = os.path.join(REFINER_CHECKPOINT, "model.safetensors")
checkpoint_state = load_file(checkpoint_path, device="cpu")
model.load_state_dict(checkpoint_state, strict=True)
del checkpoint_state
model.eval()

print(f"Loaded refiner checkpoint: {REFINER_CHECKPOINT}")

Loaded base model tensors: 390
Loaded refiner checkpoint: d:\projects\VietnameseVSR\checkpoints\refiner_p_only\checkpoint-105894


In [10]:
test_config = TrainingArguments(
    output_dir=OUTPUT_PATH,
    label_names=["labels", "label_lengths"],
    per_device_eval_batch_size=BATCH,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    dataloader_num_workers=NUM_WORKERS,
    dataloader_pin_memory=torch.cuda.is_available(),
    dataloader_persistent_workers=NUM_WORKERS > 0,
    dataloader_prefetch_factor=2 if NUM_WORKERS > 0 else None,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

In [11]:
test_collator = Collator(text_transform, "val")

trainer = CustomTrainer(
    model=model,
    args=test_config,
    train_dataset=None,
    eval_dataset=test_dataset,
    train_collator=test_collator,
    validation_collator=test_collator,
    text_transform=text_transform,
    allowed_trainable_names=("refiner.",),
)

In [12]:
test_metrics = trainer.evaluate(eval_dataset=test_dataset)
test_metrics = {
    key.replace("eval_", "test_", 1): value
    for key, value in test_metrics.items()
}
test_metrics

Training Loss,Validation Loss,Step,Wer,Refined Flip Rate,Base Wer
No log,87.901199,0,0.666545,0.047243,0.692765


{'test_loss': 87.90119934082031,
 'test_wer': 0.6665447950363159,
 'test_refined_flip_rate': 0.04724302966874022,
 'test_base_wer': 0.6927652955055237}